# Phase 4 check: class weighting for LightGBM

**Why this check:.** Only about 11.7% of labeled transactions are illicit. The GNN already weights illicit
rows in its loss (licit-to-illicit ratio, about 7.6 to 13.8 depending on the training period), but
LightGBM has always been trained without class weights. That was a deliberate starting choice on
12 Oct ("an untuned reference point"), but it was never tested: the tuning in notebook 10 did not
include class weighting. LightGBM is 60% of the final blend, and the report has to explain how class
imbalance was handled, so this notebook tests it.

**What `scale_pos_weight` does:** It multiplies the weight of every illicit row in LightGBM's loss.
With 1 (the default) every row counts equally. With 9, each illicit row counts as much as nine licit
rows, so the trees pay more attention to splits that separate illicit rows. ROC-AUC only depends on
the ranking of the scores, so this can only help if it changes which rows end up ranked higher, not
just how large the probabilities are.

**Settings tested:**

| Name | `scale_pos_weight` |
|---|---|
| none | 1 (the current model) |
| moderate | 3 |
| balanced | each fold's own licit-to-illicit ratio in its training rows (about 9 to 14), so both classes count equally in total |

Rule: keep a setting only if it raises the fold mean without lowering fold 2.
Model: default LightGBM on the 165 raw features. Validation: the three expanding-window folds.

## 1. Load

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
raw_cols = [c for c in train.columns if c.startswith("feat_")]

labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

## 2. Score each setting on the folds

In [2]:
FOLDS = [
    {"name": "fold 1", "train_last": 14, "valid_last": 21},
    {"name": "fold 2", "train_last": 21, "valid_last": 28},
    {"name": "fold 3", "train_last": 28, "valid_last": 35},
]
fold_names = [f["name"] for f in FOLDS]
SETTINGS = ["none", "moderate", "balanced"]

score_rows = []
for fold in FOLDS:
    tr = labeled[labeled["time_step"] <= fold["train_last"]]
    va = labeled[(labeled["time_step"] > fold["train_last"]) & (labeled["time_step"] <= fold["valid_last"])]

    # Licit-to-illicit ratio in this fold's training rows (used by the "balanced" setting)
    ratio = (tr["label"] == 0).sum() / (tr["label"] == 1).sum()

    for name in SETTINGS:
        if name == "none":
            weight = 1.0
        elif name == "moderate":
            weight = 3.0
        else:
            weight = ratio

        model = lgb.LGBMClassifier(random_state=SEED, verbose=-1, scale_pos_weight=weight)
        model.fit(tr[raw_cols], tr["label"])
        va_pred = model.predict_proba(va[raw_cols])[:, 1]

        row = {"setting": name, "fold": fold["name"], "weight_used": weight,
               "val_auc": roc_auc_score(va["label"], va_pred)}
        if fold["name"] == "fold 2":
            in_24 = (va["time_step"] == 24).values
            row["auc_step_24"] = roc_auc_score(va["label"][in_24], va_pred[in_24])
        score_rows.append(row)
    print(fold["name"], "done; licit-to-illicit ratio in training rows:", round(ratio, 2))

scores = pd.DataFrame(score_rows)

fold 1 done; licit-to-illicit ratio in training rows: 13.82
fold 2 done; licit-to-illicit ratio in training rows: 9.44
fold 3 done; licit-to-illicit ratio in training rows: 8.92


## 3. Results

In [3]:
results = scores.pivot(index="setting", columns="fold", values="val_auc").loc[SETTINGS]
results["mean"] = results[fold_names].mean(axis=1)
results["change_mean"] = results["mean"] - results.loc["none", "mean"]
results["change_fold2"] = results["fold 2"] - results.loc["none", "fold 2"]
results["auc_step_24"] = scores.dropna(subset=["auc_step_24"]).set_index("setting")["auc_step_24"]
results.round(4)

fold,fold 1,fold 2,fold 3,mean,change_mean,change_fold2,auc_step_24
setting,,,,,,,
none,0.9888,0.9572,0.9965,0.9808,0.0000,0.0000,0.7866
moderate,0.9914,0.9432,0.9959,0.9768,-0.0040,-0.0140,0.7384
balanced,0.9896,0.9396,0.9958,0.9750,-0.0058,-0.0176,0.7260


## Findings

- **Does class weighting raise the fold mean without lowering fold 2?** No. `scale_pos_weight = 3` lowers
  the mean from 0.9808 to 0.9768 (-0.0040) and fold 2 by 0.0140. Full balancing (each fold's
  licit-to-illicit ratio: 13.82, 9.44 and 8.92) lowers the mean to 0.9750 (-0.0058) and fold 2 by 0.0176.
  Fold 1 rises slightly with both (+0.0026 and +0.0008) and fold 3 falls slightly (-0.0006 and -0.0007).
  This is the same pattern as the tuning in notebook 10: changes from the defaults help fold 1 a little
  and hurt fold 2.
- **What happens on step 24?** It falls with weighting, from 0.787 to 0.738 (moderate) and 0.726
  (balanced), and falls more as the weighting gets stronger.
- **Decision: change the LightGBM part of the final model?** No. Neither setting passes the keep rule, so
  LightGBM stays unweighted. The final model and `e0968110.ipynb` are unchanged.
